# Simulation 2 — analysis (robustness, three branches)

The bounded-influence (Huber) arm should be **flat in outlier magnitude** and
**flat in contamination fraction until breakdown**, while `gllvm` explodes and
plain `log1p` degrades gently.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
if not os.path.exists(os.path.join(HERE, "sweep.py")):
    HERE = os.path.join(os.getcwd(), "simulations", "simulation_2")
sys.path.insert(0, HERE)
import sweep
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

df = sweep.load_results()
METHODS = [m for m in ["zqe", "zqe_huber", "gllvm"] if m in set(df.method)]
COL = {"zqe": "#1f77b4", "zqe_huber": "#2ca02c", "gllvm": "#d62728"}
LBL = {"zqe": "ZQE (log1p)", "zqe_huber": "ZQE (Huber)", "gllvm": "gllvm"}
fits = (df[df.method != "true"].drop_duplicates(["eps", "M", "rep", "method"])
        [["eps", "M", "rep", "method", "failed", "time_sec", "procrustes"]].reset_index(drop=True))
M_FIX, EPS_FIX = sweep.M_FIX, sweep.EPS_FIX
print("methods:", METHODS, "| reps:", fits.rep.nunique(),
      "| failures:", int(fits.failed.sum()), "/", len(fits))

## Flatness (magnitude) & breakdown (fraction)

Median Procrustes with IQR band, **log y-axis**. Left: vs outlier magnitude `M`
(at `eps`=`EPS_FIX`) — the bounded arm is flat, gllvm unbounded. Right: vs
contamination fraction `eps` (at `M`=`M_FIX`) — the bounded arm holds until its
breakdown point.

In [ ]:
def curve(ax, sub, xcol, logx):
    ok = sub[sub.failed == 0.0]
    for m in METHODS:
        g = (ok[ok.method == m].groupby(xcol).procrustes
             .agg(med="median", lo=lambda s: s.quantile(.25),
                  hi=lambda s: s.quantile(.75)).reset_index().sort_values(xcol))
        if g.empty: continue
        ax.plot(g[xcol], g["med"], "o-", color=COL[m], label=LBL[m])
        ax.fill_between(g[xcol], g["lo"], g["hi"], color=COL[m], alpha=0.18)
    ax.set_yscale("log")
    if logx: ax.set_xscale("log")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
mag = fits[(fits.eps == EPS_FIX) | (fits.eps == 0.0)].copy()
mag.loc[mag.eps == 0.0, "M"] = 1          # clean at M=1 so log-x shows it
curve(a1, mag, "M", logx=True)
a1.set_xlabel(r"outlier magnitude $M$  ($arepsilon$=%.2f; clean at M=1)" % EPS_FIX)
a1.set_ylabel("Procrustes error (log)")
a1.set_title("Flat in magnitude  (bounded vs unbounded influence)"); a1.legend()
epsl = fits[(fits.M == M_FIX) | (fits.eps == 0.0)]
curve(a2, epsl, "eps", logx=False)
a2.set_xlabel(r"contamination fraction $arepsilon$  (M=%d)" % M_FIX)
a2.set_title("Flat until breakdown  (fraction)"); a2.legend()
fig.suptitle("Robustness: gllvm / log1p / Huber (lower = better)")
fig.tight_layout(); plt.show()

## Failure rate (gllvm DNF under contamination)

In [ ]:
fail = (fits.groupby(["eps", "M", "method"]).failed.mean().reset_index())
for line, mask, x in [("eps line (M=%d)" % M_FIX, (fits.M==M_FIX)|(fits.eps==0), "eps"),
                      ("mag line (eps=%.2f)" % EPS_FIX, (fits.eps==EPS_FIX)|(fits.eps==0), "M")]:
    sub = fits[mask].groupby([x, "method"]).failed.mean().reset_index()
    print("---", line, "---")
    print(sub.pivot_table(index=x, columns="method", values="failed").round(2))

## Bias under contamination (loadings, eps line)

In [ ]:
truth = (df[df.method == "true"][["eps","M","rep","param","i","j","value"]]
         .rename(columns={"value": "true"}))
est = df[df.method.isin(METHODS) & (df.failed == 0.0)]
res = est.merge(truth, on=["eps","M","rep","param","i","j"], how="left")
res["resid"] = res["value"] - res["true"]
W_res = res[(res.param == "W") & ((res.M == M_FIX) | (res.eps == 0.0))]
bv = (W_res.groupby(["eps","method"])
      .agg(bias=("resid","mean"), rmse=("resid", lambda r: float(np.sqrt(np.mean(r**2))))).round(4))
display(bv.unstack("method"))

## Takeaways

`gllvm` (unbounded influence) explodes under a trace of contamination and DNFs
under heavy contamination. Plain `log1p` degrades gently. The **Huber** arm is
flat in outlier magnitude (bounded influence) and holds in fraction until its
breakdown point — robustness *by design*, at no cost to consistency (the `m₂`
centering supplies the correction). This is the constructive instance of the
paper's robustness–efficiency trade-off.

In [ ]:
summary = (fits[fits.failed == 0.0].groupby(["eps","M","method"])
           .procrustes.median().round(3).reset_index()
           .pivot_table(index=["eps","M"], columns="method", values="procrustes"))
summary